# Stage10: SPY Time-Aware Classification Baseline

This notebook runs the reusable Stage10 pipeline. It treats the final chronological block as future-like test data and records that the validation recall/alert-rate target is not met.


In [1]:
from pathlib import Path
import os, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
if Path.cwd().name == "notebooks": os.chdir("..")
ROOT=Path.cwd()
if not (ROOT/"src"/"modeling.py").is_file():
    for candidate in (ROOT,*ROOT.parents):
        if (candidate/"project"/"src"/"modeling.py").is_file(): ROOT=candidate/"project"; break
sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None
from src.modeling import BASE_FEATURES, EXTRA_FEATURES, run_baseline
from src.storage import read_df, write_df
PROCESSED=ROOT/"data"/"processed"; REPORTS=ROOT/"reports"; REPORTS.mkdir(exist_ok=True)


## 1. Fit chronological baseline


In [2]:
source=read_df(PROCESSED/"spy_feature_candidates_20260907-143336.parquet")
result=run_baseline(source)
timestamp="20260907-143336"
write_df(result["data"],PROCESSED/f"spy_modeling_frame_{timestamp}.parquet")
validation=pd.DataFrame([{**result["validation_metrics"],"threshold":result["threshold"],"cutoff":result["cutoff"],"regularization_c":result["regularization_c"]}])
test=pd.DataFrame([{**result["test_metrics"],"threshold":result["threshold"],"cutoff":result["cutoff"],"regularization_c":result["regularization_c"]}])
validation.to_csv(PROCESSED/f"model_validation_metrics_{timestamp}.csv",index=False)
test.to_csv(PROCESSED/f"model_test_metrics_{timestamp}.csv",index=False)
display(pd.concat([validation.assign(split="validation"),test.assign(split="test")]))


,accuracy,precision,recall,f1,pr_auc,alert_rate,true_negative,false_positive,false_negative,true_positive,threshold,cutoff,regularization_c,split
0,0.818000,0.210526,0.555556,0.305344,0.239382,0.190000,389,75,16,20,0.01699,0.52,0.3,validation
0,0.858283,0.242857,0.485714,0.323810,0.292812,0.139721,413,53,18,17,0.01699,0.52,0.3,test


## 2. Future-like error diagnostics and interpretation

The validation target is unmet: the selected cutoff keeps the alert rate below 20% but recall remains below 60%. This is a limitation of this baseline, not evidence that the target is acceptable.


In [3]:
test_y=result["test"]["label"].to_numpy(); test_pred=(result["test_probability"]>=result["cutoff"]).astype(int)
fig,axes=plt.subplots(1,2,figsize=(13,5))
ConfusionMatrixDisplay(confusion_matrix(test_y,test_pred),display_labels=["normal","high volatility"]).plot(ax=axes[0],colorbar=False,cmap="Blues")
axes[0].set_title("Future-test confusion matrix")
axes[1].plot(result["test"]["date"],result["test_probability"],color="#2563EB",label="risk probability")
axes[1].axhline(result["cutoff"],color="#DC2626",ls="--",label="validation-selected cutoff")
axes[1].scatter(result["test"].loc[test_y==1,"date"],result["test_probability"][test_y==1],color="#F59E0B",s=18,label="actual event")
axes[1].legend(); axes[1].set(title="Future-test probability path",xlabel="Date",ylabel="Probability")
fig.autofmt_xdate(); fig.tight_layout()
plot=REPORTS/f"spy_modeling_diagnostics_{timestamp}.png"; fig.savefig(plot,dpi=150,bbox_inches="tight"); plt.show()
assert result["train"]["date"].max()<result["validation"]["date"].min()<result["test"]["date"].min()
assert validation.loc[0,"alert_rate"]<=.20 and validation.loc[0,"recall"]<.60
print("Stage10 modeling checks passed:",plot.name)


Stage10 modeling checks passed: spy_modeling_diagnostics_20260907-143336.png


/var/folders/hy/9nh5rd9526vd43l1zms4t1lh0000gn/T/ipykernel_90385/653234940.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plot=REPORTS/f"spy_modeling_diagnostics_{timestamp}.png"; fig.savefig(plot,dpi=150,bbox_inches="tight"); plt.show()
